# SkyAware Flight Delay Prediction — Notebook 6: Model Deployment

**AAI-540 MLOps | Group 4**

This notebook:
1. Registers the trained model in **SageMaker Model Registry**
2. Creates a **Model Card** with model metadata
3. Deploys a **real-time endpoint** with DataCapture enabled
4. Tests the endpoint with sample invocations
5. Runs a **Batch Transform** job for production inference

## 1. Setup & Imports

In [1]:
import boto3
import sagemaker
import json
import time
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from sagemaker import get_execution_role, Model
from sagemaker.model_monitor import DataCaptureConfig

region = boto3.Session().region_name
role = get_execution_role()
sagemaker_session = sagemaker.Session()
sm_client = boto3.client('sagemaker', region_name=region)
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name=region)
s3_client = boto3.client('s3', region_name=region)
cw_client = boto3.client('cloudwatch', region_name=region)

PROCESSED_BUCKET = 'skyaware-processed-data'
MODEL_BUCKET = 'skyaware-model-artifacts'
MONITORING_BUCKET = 'skyaware-logs-monitoring'
ENDPOINT_NAME = 'skyaware-delay-endpoint'
MODEL_GROUP_NAME = 'SkyAwareFlightDelay'

# Load training job info
try:
    with open('training_job_info.json') as f:
        training_info = json.load(f)
    TRAINING_JOB_NAME = training_info['training_job_name']
    MODEL_ARTIFACT = training_info['model_artifact_s3']
    XGB_IMAGE_URI = training_info['xgb_image_uri']
    print(f'Loaded: {TRAINING_JOB_NAME}')
except FileNotFoundError:
    TRAINING_JOB_NAME = 'skyaware-xgboost-REPLACE'
    MODEL_ARTIFACT = f's3://{MODEL_BUCKET}/xgboost/REPLACE/output/model.tar.gz'
    XGB_IMAGE_URI = sagemaker.image_uris.retrieve('xgboost', region, '1.7-1')
    print('Using placeholder values — run NB04 first.')

print(f'Region: {region}')
print(f'Endpoint name: {ENDPOINT_NAME}')
print(f'Model group: {MODEL_GROUP_NAME}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Loaded: skyaware-xgboost-1781965796
Region: us-east-1
Endpoint name: skyaware-delay-endpoint
Model group: SkyAwareFlightDelay


## 2. Model Registry: Create Model Package Group

In [2]:
try:
    response = sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_GROUP_NAME,
        ModelPackageGroupDescription=(
            'SkyAware multi-class flight delay risk prediction models. '
            'Classes: 0=on-time, 1=minor-risk, 2=major-risk, 3=high-risk'
        ),
        Tags=[
            {'Key': 'Project', 'Value': 'SkyAware'},
            {'Key': 'Course', 'Value': 'AAI-540'},
            {'Key': 'Group', 'Value': '4'}
        ]
    )
    print(f'Created Model Package Group: {MODEL_GROUP_NAME}')
    print(f'ARN: {response["ModelPackageGroupArn"]}')
except sm_client.exceptions.ResourceInUse:
    print(f'Model Package Group already exists: {MODEL_GROUP_NAME}')
    grp = sm_client.describe_model_package_group(ModelPackageGroupName=MODEL_GROUP_NAME)
    print(f'ARN: {grp["ModelPackageGroupArn"]}')

Created Model Package Group: SkyAwareFlightDelay
ARN: arn:aws:sagemaker:us-east-1:706029888011:model-package-group/SkyAwareFlightDelay


## 3. Register Model Package

In [3]:
response = sm_client.create_model_package(
    ModelPackageGroupName=MODEL_GROUP_NAME,
    ModelPackageDescription='XGBoost 1.7-1 multi-class flight delay risk v1.0 — trained on BTS 2010-2021',
    InferenceSpecification={
        'Containers': [{
            'Image': XGB_IMAGE_URI,
            'ModelDataUrl': MODEL_ARTIFACT
        }],
        'SupportedTransformInstanceTypes': ['ml.m5.xlarge', 'ml.m5.2xlarge'],
        'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge'],
        'SupportedContentTypes': ['text/csv'],
        'SupportedResponseMIMETypes': ['text/csv']
    },
    ModelApprovalStatus='Approved',
    ModelMetrics={
        'ModelQuality': {
            'Statistics': {
                'ContentType': 'application/json',
                'S3Uri': f's3://{MODEL_BUCKET}/evaluation/metrics.json'
            }
        }
    },
    CustomerMetadataProperties={
        'WeightedF1': '0.87',
        'TrainYears': '2010-2018',
        'TestYears': '2022-2025',
        'NumClasses': '4'
    }
)

MODEL_PACKAGE_ARN = response['ModelPackageArn']
print(f'Model Package registered!')
print(f'ARN: {MODEL_PACKAGE_ARN}')

Model Package registered!
ARN: arn:aws:sagemaker:us-east-1:706029888011:model-package/SkyAwareFlightDelay/1


## 4. Create Model Card

In [5]:
model_card_content = {
    'model_overview': {
        'model_description': (
            'SkyAware XGBoost multi-class classification model for predicting flight delay risk. '
            'Trained on BTS Airline On-Time Statistics (2010-2018). '
            'Input: carrier/airport monthly aggregates + engineered features. '
            'Output: risk category 0-3.'
        ),
        'model_artifact': [MODEL_ARTIFACT],   # must be array
        'model_creator': 'AAI-540 Group 4 — SkyAware',
        'problem_type': 'MulticlassClassification'
    },
    'intended_uses': {
        'purpose_of_model': (
            'Predict monthly flight delay risk category per carrier-airport pair '
            'for airline operations planning (offline batch use case).'
        ),
        'factors_affecting_model_efficiency': (
            'COVID-19 disruptions (2020-2021) may affect accuracy; '
            'new carriers or airports not seen in training may be misclassified; '
            'sudden regulatory or route changes are not captured by historical features.'
        ),  # must be string, not array; intended_users is not a valid field
        'risk_rating': 'Medium',
        'explanations_for_risk_rating': (
            'Model is used for aggregate operational planning, not safety-critical decisions. '
            'Misclassification of high-risk as minor-risk is the primary failure mode.'
        )
    },
    'training_details': {
        # Only allowed fields: objective_function, training_observations, training_job_details
        'objective_function': 'multi:softmax — 4-class delay risk classification (XGBoost 1.7-1)',
        'training_observations': '140676'  # must be string, not int
    },
    'evaluation_details': [
        {
            'name': 'Test Set Evaluation (2022–2025)',
            'evaluation_observation': (
                'XGBoost achieved Weighted F1=1.0000, Macro F1=1.0000, Accuracy=1.0000 '
                'on 70,030 held-out test samples (2022-2025). '
                'Only 1 misclassification out of 70,030 '
                '(True=high-risk predicted as minor-risk). '
                'Near-perfect performance reflects that delay_rate, which directly encodes '
                'the target label thresholds, is included as a feature for the offline batch use case.'
            ),
            'datasets': ['s3://skyaware-processed-data/splits/test/test.csv']
        }
    ],
    'additional_information': {
        'ethical_considerations': (
            'Model may reflect historical systemic biases in delay rates by carrier or region. '
            'High-risk predictions for minority carriers should be validated before operational use. '
            'Model is intended for aggregate monthly risk only — not individual flight prediction.'
        ),
        'caveats_and_recommendations': (
            'For real-time advance prediction (before flights complete), delay_rate and cancel_rate '
            'are not yet observable and must be excluded. The model should be retrained on '
            'forward-looking features (carrier_rolling_delay_3m, airport_rolling_delay_3m, '
            'late_aircraft_roll_3m, is_covid_period) for that use case.'
        )
    }
}

try:
    sm_client.create_model_card(
        ModelCardName='SkyAware-XGBoost-v1',
        Content=json.dumps(model_card_content),
        ModelCardStatus='Draft'
    )
    print('Model Card created: SkyAware-XGBoost-v1 (Draft)')
except sm_client.exceptions.ResourceInUse:
    print('Model Card already exists. Updating...')
    current = sm_client.describe_model_card(ModelCardName='SkyAware-XGBoost-v1')
    sm_client.update_model_card(
        ModelCardName='SkyAware-XGBoost-v1',
        Content=json.dumps(model_card_content),
        ModelCardVersion=current['ModelCardVersion'] + 1
    )
    print('Model Card updated.')

Model Card created: SkyAware-XGBoost-v1 (Draft)


## 5. List Registered Models

In [2]:
packages = sm_client.list_model_packages(ModelPackageGroupName=MODEL_GROUP_NAME)
print(f'Model packages in {MODEL_GROUP_NAME}:')
for pkg in packages['ModelPackageSummaryList']:
    print(f'  Version: {pkg["ModelPackageVersion"]} | Status: {pkg["ModelApprovalStatus"]} | ARN: {pkg["ModelPackageArn"]}')

Model packages in SkyAwareFlightDelay:
  Version: 1 | Status: Approved | ARN: arn:aws:sagemaker:us-east-1:706029888011:model-package/SkyAwareFlightDelay/1


## 6. Deploy Real-Time Endpoint

In [4]:
from sagemaker.model_monitor import DataCaptureConfig

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{MONITORING_BUCKET}/data-capture/skyaware-delay-endpoint/'
)

MODEL_NAME = 'skyaware-xgboost-model'

# Delete stale endpoint config and model object to avoid name-collision errors on redeploy
for label, delete_fn, kwargs in [
    ('endpoint config', sm_client.delete_endpoint_config, {'EndpointConfigName': ENDPOINT_NAME}),
    ('model object',    sm_client.delete_model,           {'ModelName': MODEL_NAME}),
]:
    try:
        delete_fn(**kwargs)
        print(f'Deleted stale {label}: {list(kwargs.values())[0]}')
    except sm_client.exceptions.ResourceNotFound:
        pass  # already gone — fine
    except Exception as e:
        print(f'Could not delete {label}: {e}')

xgb_model = Model(
    image_uri=XGB_IMAGE_URI,
    model_data=MODEL_ARTIFACT,
    role=role,
    sagemaker_session=sagemaker_session,
    name=MODEL_NAME
)

print(f'Deploying to endpoint: {ENDPOINT_NAME}')
print(f'Instance type: ml.m5.large')
print(f'Data capture: s3://{MONITORING_BUCKET}/data-capture/')

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_config
)

print('Waiting for endpoint...')
waiter = sm_client.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={'Delay': 30, 'MaxAttempts': 20})

endpoint_status = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus']
print(f'Endpoint status: {endpoint_status}')
print(f'Endpoint name: {ENDPOINT_NAME}')

Deleted stale endpoint config: skyaware-delay-endpoint
Deleted stale model object: skyaware-xgboost-model
Deploying to endpoint: skyaware-delay-endpoint
Instance type: ml.m5.large
Data capture: s3://skyaware-logs-monitoring/data-capture/


-

-

-

-

-

-

!

Waiting for endpoint...


Endpoint status: InService
Endpoint name: skyaware-delay-endpoint


## 7. Test Endpoint Invocation

In [5]:
# Sample payloads — features in same order as training (no header, no label)
# Format: delay_rate,cancel_rate,divert_rate,avg_delay_per_flight,carrier_delay_pct,
#         weather_delay_pct,nas_delay_pct,security_delay_pct,late_aircraft_pct,
#         quarter,is_winter,is_summer,is_peak_travel,carrier_rolling_delay_3m,
#         airport_rolling_delay_3m,carrier_yoy_delay_change,carrier_enc,airport_enc,
#         arr_flights,year,month

test_payloads = [
    # on-time scenario (low delay rate, September)
    '0.10,0.01,0.00,5.2,0.30,0.15,0.35,0.01,0.19,3,0,0,0,0.09,0.10,-0.01,5,42,8500,2023,9',
    # minor-risk scenario (moderate delay, July peak)
    '0.20,0.02,0.01,12.8,0.35,0.20,0.25,0.01,0.19,3,0,1,1,0.18,0.19,0.02,12,67,12000,2023,7',
    # high-risk scenario (high delay + cancellations, winter storm)
    '0.45,0.08,0.02,28.5,0.20,0.45,0.20,0.00,0.15,1,1,0,1,0.38,0.40,0.07,3,12,6000,2024,1'
]

label_map = {0: 'on-time', 1: 'minor-risk', 2: 'major-risk', 3: 'high-risk'}

print(f'{'Scenario':<20} {'Payload Delay Rate':<22} {'Predicted Class':<20}')
print('-' * 65)

scenarios = ['on-time', 'minor-risk', 'high-risk']
for scenario, payload in zip(scenarios, test_payloads):
    try:
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='text/csv',
            Body=payload
        )
        result = response['Body'].read().decode('utf-8').strip()
        pred_class = int(float(result))
        pred_name = label_map.get(pred_class, 'unknown')
        delay_rate = payload.split(',')[0]
        print(f'{scenario:<20} {delay_rate:<22} {pred_class} ({pred_name})')
    except Exception as e:
        print(f'{scenario:<20} ERROR: {e}')

Scenario             Payload Delay Rate     Predicted Class     
-----------------------------------------------------------------


on-time              0.10                   0 (on-time)
minor-risk           0.20                   3 (high-risk)
high-risk            0.45                   3 (high-risk)


## 8. Batch Transform (Production Inference)

In [10]:
from sagemaker.estimator import Estimator

xgb_estimator = Estimator.attach(TRAINING_JOB_NAME, sagemaker_session=sagemaker_session)

batch_transformer = xgb_estimator.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{MODEL_BUCKET}/batch-production/',
    assemble_with='Line',
    accept='text/csv'
)

print('Starting production batch transform on 2022-2025 test set...')
batch_transformer.transform(
    data=f's3://{PROCESSED_BUCKET}/splits/test/test_features.csv',
    data_type='S3Prefix',
    content_type='text/csv',
    split_type='Line'
    # input_filter / join_source / output_filter omitted:
    # test_features.csv contains only feature columns (no label), so output is predictions only
)
batch_transformer.wait()
print(f'Batch transform complete!')
print(f'Output: s3://{MODEL_BUCKET}/batch-production/')


2026-06-20 14:32:51 Starting - Preparing the instances for training
2026-06-20 14:32:51 Downloading - Downloading the training image
2026-06-20 14:32:51 Training - Training image download completed. Training in progress.
2026-06-20 14:32:51 Uploading - Uploading generated training model
2026-06-20 14:32:51 Completed - Training job completed

INFO:sagemaker:Creating model with name: skyaware-xgboost-1781965796-2026-06-20-15-02-42-156


INFO:sagemaker:Creating transform job with name: skyaware-xgboost-1781965796-2026-06-20-15-02-42-821


Starting production batch transform on 2022-2025 test set...


.

...

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-20:15:07:32:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:32:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:32:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) 


2026-06-20T15:07:37.979:[sagemaker logs]: MaxConcurrentTransforms=4, MaxPayloadInMB=6, BatchStrategy=MULTI_RECORD


[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
[2026-06-20:15:07:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:39:INFO] Determined delimiter of CSV input is ','
/miniconda3/lib/python3.9/site-packages/xgboost/core.py:122: UserWarning: ntree_limit is deprecated, use `iteration_range` or model slicing instead.
  warnings.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-20:15:07:32:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:32:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:15:07:32:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  impor

## 9. CloudWatch Endpoint Metrics

In [6]:
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

metrics_to_check = [
    ('Invocations', 'Sum'),
    ('ModelLatency', 'Average'),
    ('OverheadLatency', 'Average'),
    ('Invocation5XXErrors', 'Sum'),
    ('Invocation4XXErrors', 'Sum')
]

print(f'CloudWatch metrics for endpoint: {ENDPOINT_NAME}')
print(f'Time window: {start_time.strftime("%H:%M")} - {end_time.strftime("%H:%M")} UTC')
print('-' * 55)

for metric_name, statistic in metrics_to_check:
    try:
        cw_response = cw_client.get_metric_statistics(
            Namespace='AWS/SageMaker',
            MetricName=metric_name,
            Dimensions=[
                {'Name': 'EndpointName', 'Value': ENDPOINT_NAME},
                {'Name': 'VariantName', 'Value': 'AllTraffic'}
            ],
            StartTime=start_time,
            EndTime=end_time,
            Period=3600,
            Statistics=[statistic]
        )
        datapoints = cw_response['Datapoints']
        val = datapoints[0][statistic] if datapoints else 'N/A'
        unit = 'Microseconds' if 'Latency' in metric_name else 'count'
        print(f'  {metric_name:<28} {statistic:<8}: {val}')
    except Exception as e:
        print(f'  {metric_name}: unavailable ({e})')

CloudWatch metrics for endpoint: skyaware-delay-endpoint
Time window: 05:24 - 06:24 UTC
-------------------------------------------------------
  Invocations                  Sum     : 0.0
  ModelLatency                 Average : N/A
  OverheadLatency              Average : N/A
  Invocation5XXErrors          Sum     : N/A
  Invocation4XXErrors          Sum     : N/A


## 10. Deployment Summary

| Resource | Value |
|---|---|
| Endpoint Name | `skyaware-delay-endpoint` |
| Instance Type | ml.m5.xlarge |
| Model Registry Group | `SkyAwareFlightDelay` |
| Model Approval Status | Approved |
| Data Capture | s3://skyaware-logs-monitoring/data-capture/ (100% sampling) |
| Batch Output | s3://skyaware-model-artifacts/batch-production/ |
| Model Card | SkyAware-XGBoost-v1 (Draft) |

**Next:** NB07 will configure bias monitoring and CloudWatch dashboards for this endpoint.

In [12]:
# Save deployment info for NB07
deployment_info = {
    'endpoint_name': ENDPOINT_NAME,
    'model_package_arn': MODEL_PACKAGE_ARN,
    'model_group_name': MODEL_GROUP_NAME,
    'training_job_name': TRAINING_JOB_NAME,
    'model_artifact': MODEL_ARTIFACT,
    'xgb_image_uri': XGB_IMAGE_URI,
    'data_capture_uri': f's3://{MONITORING_BUCKET}/data-capture/'
}

with open('deployment_info.json', 'w') as f:
    json.dump(deployment_info, f, indent=2)
print('Saved deployment_info.json for NB07.')
print(json.dumps(deployment_info, indent=2))

Saved deployment_info.json for NB07.
{
  "endpoint_name": "skyaware-delay-endpoint",
  "model_package_arn": "arn:aws:sagemaker:us-east-1:706029888011:model-package/SkyAwareFlightDelay/1",
  "model_group_name": "SkyAwareFlightDelay",
  "training_job_name": "skyaware-xgboost-1781965796",
  "model_artifact": "s3://skyaware-model-artifacts/xgboost/skyaware-xgboost-1781965796/output/model.tar.gz",
  "xgb_image_uri": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1",
  "data_capture_uri": "s3://skyaware-logs-monitoring/data-capture/"
}
